# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will rank content items using a simple baseline action score based on observed search opportunity and performance signals. Higher search volume and impressions indicate greater opportunity, while lower click-through rate and weaker ranking position can indicate room for improvement. The score is a prioritization tool for human review, not a prediction that a refresh will definitely improve performance.

Reason codes:
- HIGH_DEMAND: relatively high search volume
- LOW_CTR: relatively low click-through rate
- LOW_RANKING: relatively weak ranking position
- HIGH_IMPRESSIONS: relatively high impressions
- REFRESH_PRIORITY: multiple signals indicate that the content is worth reviewing

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("content_refresh_anonymized.csv")

print("Available columns:")
print(df.columns.tolist())

data = df.copy()

if "search_volume" in data.columns:
    data["search_volume"] = pd.to_numeric(
        data["search_volume"], errors="coerce"
    )

if "impressions_90d" in data.columns:
    data["impressions_90d"] = pd.to_numeric(
        data["impressions_90d"], errors="coerce"
    )

if "clicks_90d" in data.columns:
    data["clicks_90d"] = pd.to_numeric(
        data["clicks_90d"], errors="coerce"
    )

if {"clicks_90d", "impressions_90d"}.issubset(data.columns):
    data["ctr_90d"] = (
        data["clicks_90d"] /
        data["impressions_90d"].replace(0, np.nan)
    )

score_parts = []

if "search_volume" in data.columns:
    data["demand_score"] = data["search_volume"].rank(pct=True)
    score_parts.append(("demand_score", 0.40))

if "impressions_90d" in data.columns:
    data["impression_score"] = data["impressions_90d"].rank(pct=True)
    score_parts.append(("impression_score", 0.30))

if "ctr_90d" in data.columns:
    data["ctr_opportunity"] = 1 - data["ctr_90d"].rank(pct=True)
    score_parts.append(("ctr_opportunity", 0.30))

data["baseline_score"] = sum(
    data[column] * weight
    for column, weight in score_parts
)

median_search_volume = (
    data["search_volume"].median()
    if "search_volume" in data.columns
    else np.nan
)

median_impressions = (
    data["impressions_90d"].median()
    if "impressions_90d" in data.columns
    else np.nan
)

median_ctr = (
    data["ctr_90d"].median()
    if "ctr_90d" in data.columns
    else np.nan
)

def get_reason_codes(row):
    reasons = []

    if "search_volume" in data.columns:
        if row["search_volume"] >= median_search_volume:
            reasons.append("HIGH_DEMAND")

    if "impressions_90d" in data.columns:
        if row["impressions_90d"] >= median_impressions:
            reasons.append("HIGH_IMPRESSIONS")

    if "ctr_90d" in data.columns:
        if row["ctr_90d"] <= median_ctr:
            reasons.append("LOW_CTR")

    if len(reasons) >= 2:
        reasons.append("REFRESH_PRIORITY")

    return "|".join(reasons)

data["reason_code"] = data.apply(get_reason_codes, axis=1)

priority_threshold = data["baseline_score"].quantile(0.75)

data["action"] = np.where(
    data["baseline_score"] >= priority_threshold,
    "REVIEW_FOR_REFRESH",
    "LOWER_PRIORITY"
)

data["confidence_note"] = np.where(
    data["baseline_score"] >= data["baseline_score"].quantile(0.90),
    "Higher directional priority based on multiple observed signals",
    np.where(
        data["baseline_score"] >= priority_threshold,
        "Moderate directional priority; human review required",
        "Lower directional priority"
    )
)

data["what_would_make_it_wrong"] = (
    "Missing or stale data, seasonality, or factors not represented in the dataset"
)

ranked = (
    data
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1

output_cols = [
    col for col in [
        "rank",
        "content_hash_id",
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "ctr_90d",
        "baseline_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
    if col in ranked.columns
]

queue = ranked[output_cols]

output_path = Path("work/outputs")
output_path.mkdir(parents=True, exist_ok=True)

queue.to_csv(
    output_path / "baseline_action_score.csv",
    index=False
)

print(f"Ranked rows: {len(queue):,}")
print(f"Output written to: {output_path / 'baseline_action_score.csv'}")

display(queue.head(20))

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Ranked rows: 30,000
Output written to: work/outputs/baseline_action_score.csv


,rank,search_volume,impressions_90d,clicks_90d,ctr_90d,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,2900.0,16156,0,0.000000,0.907412,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
1,2,33100.0,12275,0,0.000000,0.903999,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
2,3,320.0,14519,0,0.000000,0.887071,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
3,4,210.0,16786,0,0.000000,0.883892,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
4,5,49500.0,6483,0,0.000000,0.882851,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
5,6,22200.0,6188,0,0.000000,0.880492,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
6,7,2900.0,5090,0,0.000000,0.869332,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
7,8,3600.0,4963,0,0.000000,0.869201,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
8,9,60500.0,4560,0,0.000000,0.868839,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
9,10,1900.0,5176,0,0.000000,0.868524,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 items are reviewed as a prioritized queue rather than as guaranteed refresh recommendations. Each item receives an action, reason code, confidence note, and a statement describing what could make the recommendation wrong. The review is based only on observed signals available in the dataset.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

display(
    top20[
        [
            col for col in [
                "rank",
                "content_hash_id",
                "action",
                "reason_code",
                "confidence_note",
                "what_would_make_it_wrong"
            ]
            if col in top20.columns
        ]
    ]
)

print("\nTop-20 action counts:")
print(top20["action"].value_counts())

print("\nTop-20 reason codes:")
print(top20["reason_code"].value_counts())

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
1,2,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
2,3,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
3,4,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
4,5,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
5,6,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
6,7,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
7,8,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
8,9,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."
9,10,REVIEW_FOR_REFRESH,HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_P...,Higher directional priority based on multiple ...,"Missing or stale data, seasonality, or factors..."



Top-20 action counts:
action
REVIEW_FOR_REFRESH    20
Name: count, dtype: int64

Top-20 reason codes:
reason_code
HIGH_DEMAND|HIGH_IMPRESSIONS|LOW_CTR|REFRESH_PRIORITY    20
Name: count, dtype: int64


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-ranked items may still be weak picks because a high baseline score does not prove that refreshing the content will improve performance. A page may have high demand or impressions for reasons that are not captured by the available fields, and ranking position can vary for factors outside this dataset.

I also check the feature and output columns for product flags, identifiers, private query information, and future-window fields. These fields should not be used to construct the baseline score if they reveal the outcome or information that would only be available after the decision point.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Potential weak picks:")
weak_picks = queue.tail(10)

display(
    weak_picks[
        [
            col for col in [
                "rank",
                "content_hash_id",
                "baseline_score",
                "action",
                "reason_code"
            ]
            if col in weak_picks.columns
        ]
    ]
)

print("\nLeakage check:")

leakage_terms = [
    "future",
    "next_",
    "outcome",
    "label",
    "target",
    "refresh",
    "declining",
    "product",
    "url",
    "query"
]

leakage_candidates = []

for col in df.columns:
    name = col.lower()

    if any(term in name for term in leakage_terms):
        leakage_candidates.append(col)

print("Potentially sensitive/leakage-related columns:")
print(leakage_candidates)

score_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "position_90d",
    "ctr_90d"
]

print("\nColumns actually used in baseline score:")
print(score_columns)

future_columns_used = [
    col for col in score_columns
    if any(term in col.lower() for term in ["future", "next"])
]

print("\nFuture-window columns used:")
print(future_columns_used)

assert not future_columns_used, "Future-window feature detected in baseline score."

print("\nLeakage check passed: no future-window feature is used in the baseline score.")

Potential weak picks:


,rank,baseline_score,action,reason_code
29990,29991,NaN,LOWER_PRIORITY,LOW_CTR
29991,29992,NaN,LOWER_PRIORITY,LOW_CTR
29992,29993,NaN,LOWER_PRIORITY,LOW_CTR
29993,29994,NaN,LOWER_PRIORITY,LOW_CTR
29994,29995,NaN,LOWER_PRIORITY,LOW_CTR
29995,29996,NaN,LOWER_PRIORITY,LOW_CTR
29996,29997,NaN,LOWER_PRIORITY,LOW_CTR
29997,29998,NaN,LOWER_PRIORITY,LOW_CTR
29998,29999,NaN,LOWER_PRIORITY,LOW_CTR
29999,30000,NaN,LOWER_PRIORITY,LOW_CTR



Leakage check:
Potentially sensitive/leakage-related columns:
[]

Columns actually used in baseline score:
['search_volume', 'impressions_90d', 'clicks_90d', 'position_90d', 'ctr_90d']

Future-window columns used:
[]

Leakage check passed: no future-window feature is used in the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.